# KNN Model - Attrition Prediction

## IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn import metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    make_scorer,
    classification_report,
    confusion_matrix
)

## LOAD DATA

In [ ]:
data = "../data/data_preprocess.csv"
KNN_data = pd.read_csv(data)
print("Data shape:", KNN_data.shape)
print("\nFirst 5 rows:")
print(KNN_data.head(5))

## CHECK DATA

In [ ]:
print("DATA OVERVIEW")
print(f"Total rows: {len(KNN_data)}")
print(f"Total columns: {len(KNN_data.columns)}")
print(f"Missing values:\n{KNN_data.isnull().sum().sum()}")

## CHECK TARGET IMBALANCE

In [ ]:
print("TARGET CLASS DISTRIBUTION")

print("\nClass counts:")
print(KNN_data['attrition'].value_counts())

print("\nClass proportion (%):")
print(KNN_data['attrition'].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    x=KNN_data['attrition'],
    hue=KNN_data['attrition'],
    palette='Set2',
    legend=False
)

plt.xlabel("Attrition")
plt.ylabel("Count")
plt.title("Target Class Distribution - Attrition Dataset")
plt.xticks([0, 1], ['No Attrition', 'Attrition'])

plt.show()

## SELECT X AND Y (31 FEATURES)

In [ ]:
drop_cols = ['attrition']

feature_cols = [c for c in KNN_data.columns if c not in drop_cols]

X = KNN_data[feature_cols]
y = KNN_data['attrition']

print("Number of features:", len(feature_cols))
print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

## TRAIN-TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("TRAIN-TEST SPLIT (Stratified)")
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set class distribution:\n{y_train.value_counts()}")
print(f"\nTest set class distribution:\n{y_test.value_counts()}")

## EXPERIMENT 1: KNN - FIND BEST K

In [ ]:
print("EXPERIMENT 1: KNN - FIND BEST K")

k_values = range(1, 51)
cv_scores = []
cv_f1_no_attr = []
cv_f1_attr = []

scoring = {
    'f1_macro': 'f1_macro',
    'f1_no_attrition': make_scorer(f1_score, pos_label=0),
    'f1_attrition': make_scorer(f1_score, pos_label=1)
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for k in k_values:
    knn_model = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(
            n_neighbors=k,
            weights='uniform'
        ))
    ])

    res = cross_validate(
        knn_model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    cv_scores.append(res['test_f1_macro'].mean())
    cv_f1_no_attr.append(res['test_f1_no_attrition'].mean())
    cv_f1_attr.append(res['test_f1_attrition'].mean())

best_idx = int(np.argmax(cv_scores))
best_k = k_values[best_idx]

print(f"\nBest K: {best_k}")
print(f"Best CV Macro F1:            {cv_scores[best_idx]:.4f}")
print(f"  CV F1 (No Attrition):      {cv_f1_no_attr[best_idx]:.4f}")
print(f"  CV F1 (Attrition):         {cv_f1_attr[best_idx]:.4f}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(k_values, cv_scores, marker='o', linewidth=2, markersize=6)
plt.xlabel("K (Number of Neighbors)", fontsize=12)
plt.ylabel("Cross-Validation Macro F1", fontsize=12)
plt.title("K Selection using 5-Fold Stratified Cross-Validation", fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 51, 5))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(k_values, cv_scores, marker='o', label='Macro F1', linewidth=2, markersize=5)
plt.plot(k_values, cv_f1_no_attr, marker='s', label='F1 - No Attrition', linewidth=2, markersize=5)
plt.plot(k_values, cv_f1_attr, marker='^', label='F1 - Attrition', linewidth=2, markersize=5)
plt.xlabel("K (Number of Neighbors)", fontsize=12)
plt.ylabel("Cross-Validation F1", fontsize=12)
plt.title("K Selection: Macro F1 vs F1 per Class", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 51, 5))
plt.tight_layout()
plt.show()

## EXPERIMENT 2: KNN - FIND BEST WEIGHT

In [ ]:
print("EXPERIMENT 2: KNN - FIND BEST WEIGHT")

weights_results = []

for weight in ['uniform', 'distance']:
    knn_model = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(
            n_neighbors=best_k,
            weights=weight
        ))
    ])

    res = cross_validate(
        knn_model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    weights_results.append({
        'Weight': weight,
        'CV_Macro_F1': res['test_f1_macro'].mean(),
        'CV_F1_No_Attrition': res['test_f1_no_attrition'].mean(),
        'CV_F1_Attrition': res['test_f1_attrition'].mean(),
        'CV_Macro_F1_Std': res['test_f1_macro'].std()
    })

weights_results_df = pd.DataFrame(weights_results)
print("\nWeights Comparison:")
print(weights_results_df)

best_weight = weights_results_df.loc[
    weights_results_df['CV_Macro_F1'].idxmax(),
    'Weight'
]

print(f"\nBest K: {best_k}")
print(f"Best Weight: {best_weight}")

## SELECT FINAL MODEL

In [ ]:
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(
        n_neighbors=best_k,
        weights=best_weight
    ))
])

final_res = cross_validate(
    final_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

final_cv = final_res['test_f1_macro']
final_cv_f1_no_attr = final_res['test_f1_no_attrition']
final_cv_f1_attr = final_res['test_f1_attrition']

print("SELECT FINAL MODEL")
print(f"\nSelected: KNN (no resampling, {len(feature_cols)} features)")
print(f"  Configuration: K={best_k}, Weight={best_weight}")
print(f"  CV Macro F1:           {final_cv.mean():.4f} (±{final_cv.std():.4f})")
print(f"  CV F1 (No Attrition):  {final_cv_f1_no_attr.mean():.4f} (±{final_cv_f1_no_attr.std():.4f})")
print(f"  CV F1 (Attrition):     {final_cv_f1_attr.mean():.4f} (±{final_cv_f1_attr.std():.4f})")

## FINAL TEST SET EVALUATION

In [ ]:
final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
y_prob = final_pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label=1)
f1_no_attr = f1_score(y_test, y_pred, pos_label=0)
macro_f1 = f1_score(y_test, y_pred, average='macro')
balanced_acc = balanced_accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"\nTest Set Metrics:")
print(f"  Accuracy:           {accuracy:.4f}")
print(f"  Precision:          {precision:.4f}")
print(f"  Recall:             {recall:.4f}")
print(f"  F1 (No Attrition):  {f1_no_attr:.4f}")
print(f"  F1 (Attrition):     {f1:.4f}")
print(f"  Macro F1:           {macro_f1:.4f}")
print(f"  Balanced Accuracy:  {balanced_acc:.4f}")
print(f"  ROC-AUC:            {roc_auc:.4f}")

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No Attrition', 'Attrition']))

In [ ]:
f1_per_class = pd.DataFrame({
    'Class': ['No Attrition', 'Attrition', 'Macro Avg'],
    'F1-score': [f1_no_attr, f1, macro_f1]
})

print(f1_per_class.to_string(index=False))

plt.figure(figsize=(7, 5))
ax = sns.barplot(
    data=f1_per_class,
    x='Class',
    y='F1-score',
    hue='Class',
    palette='Set2',
    legend=False
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f')
plt.ylim(0, 1)
plt.title("F1-score per Class vs Macro F1 (Test Set)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

In [ ]:
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix=conf_matrix,
    display_labels=['No Attrition', 'Attrition']
)

fig, ax = plt.subplots(figsize=(8, 6))
cm_display.plot(cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix - KNN (Test Set)', fontsize=14)
plt.tight_layout()
plt.show()

## EXPORT RESULTS FOR POWER BI

In [ ]:
k_results = pd.DataFrame({
    'K': list(k_values),
    'CV_Macro_F1': cv_scores,
    'CV_F1_No_Attrition': cv_f1_no_attr,
    'CV_F1_Attrition': cv_f1_attr
})

k_results.to_csv('knn_k_selection.csv', index=False)

In [ ]:
evaluation_results = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1-score (No Attrition)',
        'F1-score (Attrition)',
        'Macro F1',
        'Balanced Accuracy',
        'ROC-AUC'
    ],
    'Score': [
        accuracy,
        precision,
        recall,
        f1_no_attr,
        f1,
        macro_f1,
        balanced_acc,
        roc_auc
    ]
})

evaluation_results.to_csv('knn_evaluation.csv', index=False)

In [ ]:
cm_results = pd.DataFrame({
    'Actual': [
        'No Attrition',
        'No Attrition',
        'Attrition',
        'Attrition'
    ],
    'Predicted': [
        'No Attrition',
        'Attrition',
        'No Attrition',
        'Attrition'
    ],
    'Count': [
        conf_matrix[0, 0],
        conf_matrix[0, 1],
        conf_matrix[1, 0],
        conf_matrix[1, 1]
    ]
})

cm_results.to_csv('knn_confusion_matrix.csv', index=False)

In [ ]:
summary_results = pd.DataFrame({
    'Model_Type': ['KNN'],
    'N_Features': [len(feature_cols)],
    'Best_K': [best_k],
    'Best_Weight': [best_weight],
    'CV_Macro_F1': [final_cv.mean()],
    'CV_F1_No_Attrition': [final_cv_f1_no_attr.mean()],
    'CV_F1_Attrition': [final_cv_f1_attr.mean()],
    'Test_Accuracy': [round(accuracy * 100, 2)],
    'Test_Macro_F1': [round(macro_f1 * 100, 2)],
    'Test_F1_No_Attrition': [round(f1_no_attr * 100, 2)],
    'Test_F1_Attrition': [round(f1 * 100, 2)],
    'Test_ROC_AUC': [round(roc_auc * 100, 2)]
})

summary_results.to_csv('knn_summary.csv', index=False)

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    output_dict=True
)

performance_results = pd.DataFrame({
    'Class': [
        'No Attrition',
        'Attrition'
    ],
    'Precision': [
        report['0']['precision'],
        report['1']['precision']
    ],
    'Recall': [
        report['0']['recall'],
        report['1']['recall']
    ],
    'F1-score': [
        report['0']['f1-score'],
        report['1']['f1-score']
    ],
    'Support': [
        int(report['0']['support']),
        int(report['1']['support'])
    ]
})

performance_results.to_csv('knn_performance.csv', index=False)

In [ ]:
weight_comparison = pd.DataFrame({
    'Configuration': [
        'Uniform',
        'Distance'
    ],
    'CV_Macro_F1': [
        weights_results_df.loc[weights_results_df['Weight'] == w, 'CV_Macro_F1'].iloc[0]
        for w in ['uniform', 'distance']
    ],
    'CV_F1_No_Attrition': [
        weights_results_df.loc[weights_results_df['Weight'] == w, 'CV_F1_No_Attrition'].iloc[0]
        for w in ['uniform', 'distance']
    ],
    'CV_F1_Attrition': [
        weights_results_df.loc[weights_results_df['Weight'] == w, 'CV_F1_Attrition'].iloc[0]
        for w in ['uniform', 'distance']
    ]
})

weight_comparison.to_csv('knn_weight_comparison.csv', index=False)

## MODEL LIMITATIONS AND ETHICAL USE

In [ ]:
print("""
⚠️ ข้อจำกัดของโมเดลและการใช้งานอย่างมีจริยธรรม:
- โมเดลใช้เพื่อระบุ "ความสัมพันธ์" ของปัจจัยต่าง ๆ กับการลาออก ไม่ได้ยืนยันความเป็นเหตุและผล
- ไม่สามารถระบุได้ว่า "การดำเนินการใดจะช่วยลดการลาออกได้"
- ไม่ควรนำโมเดลไปใช้ในการให้คะแนนพนักงานรายบุคคล
- ผลการทำนายแสดง "รูปแบบความเสี่ยง" ไม่ใช่ข้อสรุปหรือผลลัพธ์ที่แน่นอน

การใช้งานที่เหมาะสม:
✅ วิเคราะห์แผนกหรือตำแหน่งงานที่มีรูปแบบความเสี่ยงสูง
✅ ติดตามแนวโน้มด้านความผูกพันและการมีส่วนร่วมของพนักงานในแต่ละทีม
✅ ใช้เป็นข้อมูลประกอบเพื่อให้ฝ่าย HR ตรวจสอบปัจจัยที่เกี่ยวข้องเพิ่มเติม

การใช้งานที่ไม่เหมาะสม:
❌ ใช้ตัดสินใจเลื่อนตำแหน่งหรือเลิกจ้างพนักงาน
❌ สรุปว่าปัจจัยใดเป็นสาเหตุโดยตรง เช่น ลดจำนวนวันที่ต้องเข้าออฟฟิศแล้วจะลดการลาออก
❌ ให้คะแนนหรือจัดอันดับพนักงานเป็นรายบุคคล
""")